# 04 对称正定矩阵的平方根分解

对称正定矩阵满足 $A=A^T$ 且 $\boldsymbol{x}^TA\boldsymbol{x}>0$。这类矩阵可以使用 Cholesky 分解和 $LDL^T$ 分解，比一般 LU 更节省计算和存储。


In [ ]:
import pathlib
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
CHAPTER = ROOT / "chapters" / "ch06_direct_linear_systems"
SCRIPTS = CHAPTER / "scripts"
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from py_sc import cholesky_factorization, cholesky_solve, ldlt_factorization, ldlt_solve, lu_doolittle
from py_sc.direct_linear import relative_residual


## Cholesky 分解

Cholesky 分解写作

$$
A=LL^T,
$$

其中 $L$ 为下三角矩阵。递推公式为

$$
l_{jj}=\sqrt{a_{jj}-\sum_{k=1}^{j-1}l_{jk}^2},
$$

$$
l_{ij}=\frac{a_{ij}-\sum_{k=1}^{j-1}l_{ik}l_{jk}}{l_{jj}}.
$$


In [ ]:
def teaching_cholesky(A):
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    L = np.zeros_like(A)
    for j in range(n):
        value = A[j, j] - L[j, :j] @ L[j, :j]
        if value <= 0:
            raise ValueError("不是正定矩阵")
        L[j, j] = np.sqrt(value)
        for i in range(j + 1, n):
            L[i, j] = (A[i, j] - L[i, :j] @ L[j, :j]) / L[j, j]
    return L

R = np.array([[2.0, -1.0, 0.5], [0.0, 1.5, 2.0], [1.0, 0.0, 1.0]])
A = R.T @ R + np.eye(3)
L = teaching_cholesky(A)
print("教学版重构误差:", np.linalg.norm(A - L @ L.T))
print("与 NumPy cholesky 的差:", np.linalg.norm(L - np.linalg.cholesky(A)))


## Cholesky 求解

分解后求解

$$
L\boldsymbol{y}=\boldsymbol{b},
\qquad
L^T\boldsymbol{x}=\boldsymbol{y}.
$$


In [ ]:
x_exact = np.array([1.0, -2.0, 0.5])
b = A @ x_exact
formal_L = cholesky_factorization(A)
x = cholesky_solve(formal_L, b)
print("解:", x)
print("残差:", relative_residual(A, x, b))


## $LDL^T$ 分解

$LDL^T$ 分解写作

$$
A=LDL^T,
$$

其中 $L$ 是单位下三角矩阵，$D$ 是对角矩阵。它与 Cholesky 的关系是：若 $D$ 的对角元均为正，则可令 $L_{chol}=L\sqrt{D}$。

这种形式不显式计算平方根，适合展示对称分解的结构。


In [ ]:
def teaching_ldlt(A):
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    L = np.eye(n)
    D = np.zeros(n)
    for j in range(n):
        D[j] = A[j, j] - np.sum(L[j, :j] ** 2 * D[:j])
        if D[j] <= 0:
            raise ValueError("不是正定矩阵")
        for i in range(j + 1, n):
            L[i, j] = (A[i, j] - np.sum(L[i, :j] * L[j, :j] * D[:j])) / D[j]
    return L, D

L_ldlt, D = teaching_ldlt(A)
print("LDLT 重构误差:", np.linalg.norm(A - L_ldlt @ np.diag(D) @ L_ldlt.T))
x_ldlt = ldlt_solve(L_ldlt, D, b)
print("LDLT 求解残差:", relative_residual(A, x_ldlt, b))


## 非正定失败情形

Cholesky 和正定 $LDL^T$ 都依赖正定性。若矩阵只是对称但不正定，递推中会出现非正对角量。


In [ ]:
A_indefinite = np.array([[1.0, 2.0], [2.0, 1.0]])
for name, factor in [("Cholesky", cholesky_factorization), ("LDLT", ldlt_factorization)]:
    try:
        factor(A_indefinite)
    except ValueError as exc:
        print(f"{name} 失败:", exc)


## 小结

* 对称正定矩阵应优先考虑 Cholesky 或 $LDL^T$。
* Cholesky 使用平方根，计算量约为一般 LU 的一半。
* $LDL^T$ 不显式计算平方根，但正定矩阵中 $D$ 的对角元应为正。
* 对称不定矩阵需要带主元的专门分解，本章只作为拓展入口。
